In [21]:
import os
import gc
import math
import nltk
from collections import Counter
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import torch
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [23]:
nltk.download('words', quiet=True)
from nltk.corpus import words
english_dict = set(w.lower() for w in words.words() if 4 <= len(w) <= 15)
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/SIH_dataset/dga_domains_full.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:

print("Loading full dataset...")
df = pd.read_csv(file_path, header=None, names=['class', 'subclass', 'domain'], dtype=str)

df['class'] = df['class'].str.strip().str.lower()
df['subclass'] = df['subclass'].str.strip()
df['domain'] = df['domain'].astype(str).str.strip().str.lower()

# Binary Target Mapping: 1 for DGA, 0 for Benign
df['target'] = (df['class'] == 'dga').astype(np.uint8)

print(f"Loaded {len(df):,} domains.")

Loading full dataset...
Loaded 674,898 domains.


In [25]:
def calculate_shannon_entropy(s):
    """Calculates mathematical randomness (entropy) of the string[cite: 3]."""
    if not s: return 0.0
    length = len(s)
    counts = Counter(s)
    return -sum((cnt / length) * math.log2(cnt / length) for cnt in counts.values())
def fast_dict_match_ratio(sld):
    """Scans internal substrings against the English dictionary hash set instantly."""
    if not sld: return 0.0
    n = len(sld)
    matched = [False] * n

    for length in range(4, min(n + 1, 16)):
        for i in range(n - length + 1):
            sub = sld[i:i + length]
            if sub in english_dict:
                for idx in range(i, i + length):
                    matched[idx] = True
    return sum(matched) / max(n, 1)
def fast_vowel_consonant_ratio(sld):
    """Calculates the ratio of consonants to vowels."""
    vowels = sum(1 for c in sld if c in 'aeiou')
    consonants = sum(1 for c in sld if c.isalpha() and c not in 'aeiou')
    return consonants / max(vowels, 1)


In [26]:
print("Extracting lexical and NLP features...")

X_base = pd.DataFrame(index=df.index)
sld_series = df['domain'].apply(lambda x: x.split('.')[0] if '.' in x else x)

# A. Length & Entropy
X_base['len_full'] = df['domain'].str.len().astype(np.uint16)
X_base['len_sld'] = sld_series.str.len().astype(np.uint16)
X_base['entropy'] = sld_series.apply(calculate_shannon_entropy).astype(np.float32)

# B. TLD Profiling
high_risk_tlds = ('.cc', '.ru', '.biz', '.info', '.top', '.ddns.net', '.xyz', '.ws')
X_base['tld_len'] = df['domain'].apply(lambda x: len(x.split('.')[-1]) if '.' in x else 0).astype(np.uint8)
X_base['high_risk_tld'] = df['domain'].str.endswith(high_risk_tlds).astype(np.uint8)

# C. Character Ratios
X_base['cv_ratio'] = sld_series.apply(fast_vowel_consonant_ratio).astype(np.float32)
X_base['digit_ratio'] = sld_series.apply(lambda s: sum(1 for c in s if c.isdigit()) / max(len(s), 1)).astype(np.float32)
X_base['dict_match_ratio'] = sld_series.apply(fast_dict_match_ratio).astype(np.float32)

# D. Fast Character N-Grams (NLP Feature)[cite: 3]
# Restricted to top 15 bigrams to keep Colab memory low and speed high
print("Computing lightweight character n-grams...")
tfidf = TfidfVectorizer(analyzer='char', ngram_range=(2, 2), max_features=15, lowercase=True)
ngram_matrix = tfidf.fit_transform(sld_series).astype(np.float32)
ngram_df = pd.DataFrame(ngram_matrix.toarray(), columns=[f"ngram_{f}" for f in tfidf.get_feature_names_out()])

# Combine features
X = pd.concat([X_base, ngram_df], axis=1)
y = df['target'].values

del df, X_base, ngram_df, sld_series
gc.collect()


Extracting lexical and NLP features...
Computing lightweight character n-grams...


34

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"\nTraining Set: {X_train.shape[0]:,} | Testing Set: {X_test.shape[0]:,}")
tree_method = 'hist'
if torch.cuda.is_available():
    print("GPU is available for PyTorch, but 'gpu_hist' is not a valid XGBoost tree method. Using 'hist'.")
else:
    print("GPU is not available. Using CPU 'hist' for XGBoost.")



Training Set: 539,918 | Testing Set: 134,980
GPU is available for PyTorch, but 'gpu_hist' is not a valid XGBoost tree method. Using 'hist'.


In [28]:
xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=12,
    learning_rate=0.1,
    scale_pos_weight=1.5,
    tree_method=tree_method,
    random_state=42,
    n_jobs=-1
)
print("\nTraining XGBoost Classifier...")
xgb_model.fit(X_train, y_train)


Training XGBoost Classifier...


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=12,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=150,
              n_jobs=-1, num_parallel_tree=None, ...)

In [29]:
y_probs = xgb_model.predict_proba(X_test)[:, 1]
best_thresh = 0.50
max_f1 = 0.0
print("\nScanning thresholds (0.50 - 0.75) targeting FPR < 5%...")
for thresh in np.arange(0.50, 0.76, 0.02):
    preds = (y_probs >= thresh).astype(int)
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    if fpr <= 0.05 and f1 > max_f1:
        max_f1 = f1
        best_thresh = thresh

print(f"Selected Optimal Threshold: {best_thresh:.2f}")



Scanning thresholds (0.50 - 0.75) targeting FPR < 5%...
Selected Optimal Threshold: 0.68


In [30]:
y_pred_final = (y_probs >= best_thresh).astype(int)
cm_final = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm_final.ravel()
print("\n" + "=" * 65)
print(f"XGBOOST DGA EVALUATION METRICS (Threshold = {best_thresh:.2f})")
print("=" * 65)
print(classification_report(y_test, y_pred_final, target_names=['Benign (0)', 'DGA (1)'], digits=4))
print(f"ROC-AUC Score:        {roc_auc_score(y_test, y_probs):.4f}")
print(f"False Positive Rate:  {fp / (fp + tn):.4%}")
print("\nConfusion Matrix:")
display(pd.DataFrame(
    cm_final,
    index=['Actual Benign (0)', 'Actual DGA (1)'],
    columns=['Predicted Benign (0)', 'Predicted DGA (1)']
))



XGBOOST DGA EVALUATION METRICS (Threshold = 0.68)
              precision    recall  f1-score   support

  Benign (0)     0.8648    0.9526    0.9066     67480
     DGA (1)     0.9473    0.8511    0.8966     67500

    accuracy                         0.9019    134980
   macro avg     0.9060    0.9019    0.9016    134980
weighted avg     0.9061    0.9019    0.9016    134980

ROC-AUC Score:        0.9669
False Positive Rate:  4.7377%

Confusion Matrix:


,Predicted Benign (0),Predicted DGA (1)
Actual Benign (0),64283,3197
Actual DGA (1),10049,57451


In [ ]:
output_dir = "/content/C2_EDA"
os.makedirs(output_dir, exist_ok=True)
model_path = os.path.join(output_dir, "dga_xgboost_optimized.pkl")
joblib.dump(xgb_model, model_path)
print(f"\nModel saved to: {model_path}")